# Train conditional spectrogram VAE

In [ ]:
# Colab setup. Run from the repo root after uploading/cloning this project.
# !pip install -r requirements.txt

# ==== Colab project setup ====

from pathlib import Path
import sys
import os

REPO_URL = "https://github.com/vadatta/spectogram-audio.git"
REPO_NAME = "spectogram-audio"

# Go to Colab workspace
%cd /content

# Clone repo if it does not exist yet
if not Path(REPO_NAME).exists():
    !git clone {REPO_URL}

# Enter project folder
%cd /content/{REPO_NAME}/audio

# Pull latest changes from GitHub
!git pull

# Add project root to Python import path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Files:", os.listdir(PROJECT_ROOT))
!ls

In [ ]:
!python -m pip install -r requirements.txt

In [ ]:
import sys
print(sys.executable)

In [ ]:
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from nsynth_pipeline.data import MelConfig, NOTE_NAMES, make_nsynth_loaders
from nsynth_pipeline.models import ConditionalSpectrogramVAE
from nsynth_pipeline.train_utils import kl_divergence, move_batch, save_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

In [ ]:
DATASET_NAME = "jg583/NSynth"
BATCH_SIZE = 32
NUM_WORKERS = 2
EPOCHS = 10
BETA = 1e-4

# Set these to small integers for smoke tests; keep None for full train/test splits.
MAX_TRAIN_ITEMS = None
MAX_TEST_ITEMS = None

CHECKPOINT_PATH = "checkpoints/conditional_spectrogram_vae.pt"
mel_config = MelConfig()

In [ ]:
# In Colab, this stores the Hugging Face dataset cache on Google Drive so it persists across runtimes.
HF_CACHE_DIR = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    HF_CACHE_DIR = "/content/drive/MyDrive/hf_datasets_cache"
except ModuleNotFoundError:
    print("Not running in Colab; using the local Hugging Face cache.")

print("HF cache dir:", HF_CACHE_DIR)

In [ ]:
loaders = make_nsynth_loaders(
    dataset_name=DATASET_NAME,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    max_train_items=MAX_TRAIN_ITEMS,
    max_test_items=MAX_TEST_ITEMS,
    mel_config=mel_config,
    cache_dir=HF_CACHE_DIR,
)
batch = next(iter(loaders["train"]))
input_shape = tuple(batch["spectrogram"].shape[1:])
input_shape

In [ ]:
vae = ConditionalSpectrogramVAE(input_shape=input_shape, latent_dim=128).to(device)
optimizer = torch.optim.AdamW(vae.parameters(), lr=1e-3, weight_decay=1e-5)

The VAE is conditioned on the ground-truth raw pitch during training, not the classifier prediction.

## Training code

In [ ]:
def train_vae_one_epoch(model, loader, optimizer, device, beta=1e-4):
    model.train()
    running_loss = 0.0
    running_mse = 0.0
    running_kl = 0.0
    total = 0

    for batch in tqdm(loader, desc="vae train"):
        batch = move_batch(batch, device)
        spectrogram = batch["spectrogram"]
        pitch = batch["pitch"]

        optimizer.zero_grad(set_to_none=True)
        reconstruction, mu, logvar = model(spectrogram, pitch)
        mse = F.mse_loss(reconstruction, spectrogram)
        kl = kl_divergence(mu, logvar)
        loss = mse + beta * kl
        loss.backward()
        optimizer.step()

        batch_size = spectrogram.shape[0]
        running_loss += loss.item() * batch_size
        running_mse += mse.item() * batch_size
        running_kl += kl.item() * batch_size
        total += batch_size

    return {"loss": running_loss / total, "mse": running_mse / total, "kl": running_kl / total}


@torch.no_grad()
def evaluate_vae(model, loader, device, beta=1e-4):
    model.eval()
    running_loss = 0.0
    running_mse = 0.0
    running_kl = 0.0
    total = 0

    for batch in tqdm(loader, desc="vae eval"):
        batch = move_batch(batch, device)
        spectrogram = batch["spectrogram"]
        pitch = batch["pitch"]
        reconstruction, mu, logvar = model(spectrogram, pitch)
        mse = F.mse_loss(reconstruction, spectrogram)
        kl = kl_divergence(mu, logvar)
        loss = mse + beta * kl

        batch_size = spectrogram.shape[0]
        running_loss += loss.item() * batch_size
        running_mse += mse.item() * batch_size
        running_kl += kl.item() * batch_size
        total += batch_size

    return {"loss": running_loss / total, "mse": running_mse / total, "kl": running_kl / total}

In [ ]:
for epoch in range(1, EPOCHS + 1):
    train_metrics = train_vae_one_epoch(vae, loaders["train"], optimizer, device, beta=BETA)
    test_metrics = evaluate_vae(vae, loaders["test"], device, beta=BETA)
    print({"epoch": epoch, "train": train_metrics, "test": test_metrics})
    save_checkpoint(
        CHECKPOINT_PATH,
        model_state_dict=vae.state_dict(),
        input_shape=input_shape,
        latent_dim=vae.latent_dim,
        beta=BETA,
        mel_config=mel_config,
        note_names=NOTE_NAMES,
    )